# Feature Engineering & Data Preparation

**Objective:** Transform the cleaned dataset into machine-learning-ready feature matrices while adhering to clinical best practices and preventing data leakage.

**Key Tasks in this Notebook:**
1. **Domain-Specific Feature Creation:** Blood pressure metrics, adiposity indicators, and lifestyle risk scores.
2. **Mitigating Data Leakage (Strategic Feature Sets):**
   - **Model A (Pre-Screening / Non-Invasive):** Excludes `fasting_blood_sugar` and `hba1c_level`. Designed for clinic triage (deciding who needs testing).
   - **Model B (Full Clinical):** Includes blood lab markers for diagnostic risk grading.
3. **Categorical Encoding:** One-hot encoding for nominal variables, binary encoding for binary flags.
4. **Target Encoding:** Multiclass mapping (`Low`: 0, `Moderate`: 1, `High`: 2) and Binary triage mapping (`High Risk`: 1, `Other`: 0).
5. **Stratified Train-Test Split:** Split before scaling to preserve class ratios and prevent data leakage.
6. **Feature Scaling:** Fit `StandardScaler` on training data only and transform test data.
7. **Export:** Save processed splits for model training in `06_modeling.ipynb`.

---
## 1. Load Cleaned Data & Initial Setup

In [ ]:
import pandas as pd
import numpy as np
import os

df = pd.read_csv('../data/diabetes_risk_cleaned.csv')
print(f"Loaded dataset: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

---
## 2. Domain-Specific Feature Engineering

We create health and lifestyle indicators grounded in clinical literature:

1. **`pulse_pressure`**: $\text{Systolic} - \text{Diastolic}$. Elevated pulse pressure reflects arterial stiffness and vascular risk.
2. **`is_hypertensive`**: Binary indicator if Systolic $\ge 140$ or Diastolic $\ge 90$ mmHg (clinical stage-2 hypertension).
3. **`bmi_category`**: Standard WHO categories (Underweight, Normal, Overweight, Obese).
4. **`high_waist_risk`**: Central adiposity flag (waist $\ge 90$ cm).
5. **`sleep_category`**: Short ($<6$h), Normal ($6-8$h), Long ($>8$h). Disrupted sleep correlates with insulin resistance.
6. **`sedentary_high_bmi`**: Interaction flag for patients who are both sedentary and overweight/obese.
7. **`lifestyle_risk_score`**: Cumulative count of negative lifestyle factors (Sedentary + Current Smoker + High Stress [$\ge 7$] + Poor Sleep).

In [ ]:
# Copy to avoid mutating original df
data = df.copy()

# 1. Blood Pressure Features
data['pulse_pressure'] = data['blood_pressure_systolic'] - data['blood_pressure_diastolic']
data['is_hypertensive'] = ((data['blood_pressure_systolic'] >= 140) | (data['blood_pressure_diastolic'] >= 90)).astype(int)

# 2. Body Composition
data['bmi_category'] = pd.cut(
    data['bmi'], 
    bins=[0, 18.5, 24.99, 29.99, 100], 
    labels=['Underweight', 'Normal', 'Overweight', 'Obese']
)
data['high_waist_risk'] = (data['waist_circumference_cm'] >= 90).astype(int)

# 3. Sleep & Stress Categories
data['sleep_category'] = pd.cut(
    data['hours_sleep_per_night'], 
    bins=[0, 5.99, 8.01, 24], 
    labels=['Short', 'Normal', 'Long']
)

# 4. Interaction & Composite Lifestyle Risk Index
data['sedentary_high_bmi'] = (
    (data['physical_activity_level'] == 'Sedentary') & (data['bmi'] >= 25)
).astype(int)

data['lifestyle_risk_score'] = (
    (data['physical_activity_level'] == 'Sedentary').astype(int) +
    (data['smoking_status'] == 'Current').astype(int) +
    (data['stress_level'] >= 7).astype(int) +
    (data['hours_sleep_per_night'] < 6).astype(int)
)

print(f"Engineered features added. New shape: {data.shape}")
data[['pulse_pressure', 'is_hypertensive', 'bmi_category', 'sleep_category', 'lifestyle_risk_score']].head()

---
## 3. Define Strategic Feature Sets (Preventing Clinical Leakage)

> **Why this matters for your clinic stakeholder:**
> If a clinic already tested `fasting_blood_sugar` and `hba1c_level`, they already know if the patient has diabetes. A model using those tests is circular for pre-screening.
>
> We create **two distinct modeling pipelines**:
> - **Pre-Screening Feature Set (Non-Invasive):** Age, BMI, waist, vitals, lifestyle, family history. **Use case:** Deciding who to invite for lab tests.
> - **Full Clinical Feature Set:** Pre-screening features + `fasting_blood_sugar` + `hba1c_level`. **Use case:** Comprehensive risk tiering once labs are available.

In [ ]:
# Drop uninformative row identifier
if 'patient_id' in data.columns:
    data = data.drop(columns=['patient_id'])

leakage_features = ['fasting_blood_sugar', 'hba1c_level']

print(f"Target column: diabetes_risk")
print(f"Leakage features flagged: {leakage_features}")

---
## 4. Encode Categorical Features

We convert non-numeric features into representations algorithms can interpret:
- **Binary Encoding:** `family_history_diabetes` (Yes $\rightarrow$ 1, No $\rightarrow$ 0)
- **One-Hot Encoding:** Nominal columns (`gender`, `city`, `physical_activity_level`, `diet_type`, `smoking_status`, `alcohol_consumption`, `income_bracket`, `bmi_category`, `sleep_category`).

In [ ]:
# Binary map
data['family_history_diabetes'] = (data['family_history_diabetes'] == 'Yes').astype(int)

# Categorical columns for one-hot encoding
nominal_cols = [
    'gender', 'city', 'physical_activity_level', 'diet_type', 
    'smoking_status', 'alcohol_consumption', 'income_bracket',
    'bmi_category', 'sleep_category'
]

# One-Hot Encoding with dummy_na=False since NaNs are already 'Unknown'
data_encoded = pd.get_dummies(data, columns=nominal_cols, drop_first=True, dtype=int)

print(f"Encoded dataset shape: {data_encoded.shape}")

---
## 5. Target Variable Formats

We prepare two versions of the target:
1. **`target_multiclass`**: Ordinal mapping (`Low`: 0, `Moderate`: 1, `High`: 2).
2. **`target_binary`**: Binary triage mapping (`High Risk`: 1, `Low/Moderate`: 0), directly serving the clinic's screening prioritization question.

In [ ]:
target_multiclass_map = {'Low': 0, 'Moderate': 1, 'High': 2}
y_multiclass = data_encoded['diabetes_risk'].map(target_multiclass_map)
y_binary = (data_encoded['diabetes_risk'] == 'High').astype(int)

print("Multiclass target counts:")
print(y_multiclass.value_counts())
print("\nBinary triage target counts (1=High Risk):")
print(y_binary.value_counts())

---
## 6. Train-Test Split (Stratified to Prevent Leakage)

> **Industry Rule:** Always split into training and test sets **before** fitting scalers or estimating feature statistics. We use `stratify=y` to preserve the exact class ratio across both splits.

In [ ]:
from sklearn.model_selection import train_test_split

# Feature matrix excluding the target string
X_all = data_encoded.drop(columns=['diabetes_risk'])

# Non-invasive feature matrix (excluding lab leakage)
X_prescreen = X_all.drop(columns=leakage_features)

# Split: 80% train, 20% test, stratified by the multiclass target
X_train, X_test, y_train, y_test = train_test_split(
    X_prescreen, y_multiclass, test_size=0.20, random_state=42, stratify=y_multiclass
)

# Also create binary targets for the split
y_train_bin = (y_train == 2).astype(int)
y_test_bin = (y_test == 2).astype(int)

print(f"X_train shape: {X_train.shape} | X_test shape: {X_test.shape}")
print(f"y_train distribution:\n{y_train.value_counts(normalize=True).round(3)}")

---
## 7. Feature Scaling (StandardScaler)

Distance-based and linear models (Logistic Regression, SVM, KNN, Neural Nets) require continuous features to be on the same scale.

**Crucial Step:** We fit the scaler **only** on `X_train` and transform both `X_train` and `X_test`.

In [ ]:
from sklearn.preprocessing import StandardScaler

continuous_cols = [
    'age', 'bmi', 'hours_sleep_per_night', 'stress_level',
    'blood_pressure_systolic', 'blood_pressure_diastolic',
    'waist_circumference_cm', 'pulse_pressure', 'lifestyle_risk_score'
]

scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

# Fit on train, transform both
X_train_scaled[continuous_cols] = scaler.fit_transform(X_train[continuous_cols])
X_test_scaled[continuous_cols] = scaler.transform(X_test[continuous_cols])

print("Continuous features successfully scaled.")
X_train_scaled[continuous_cols].head()

---
## 8. Export Processed Datasets

We save the train and test splits to `data/processed/` so our modeling notebook (`05_modeling.ipynb`) can immediately load clean, standardized inputs.

In [ ]:
os.makedirs('../data/processed', exist_ok=True)

# Unscaled (ideal for tree-based models like Random Forest, XGBoost, LightGBM)
X_train.to_csv('../data/processed/X_train_unscaled.csv', index=False)
X_test.to_csv('../data/processed/X_test_unscaled.csv', index=False)

# Scaled (ideal for Logistic Regression, SVM, Neural Nets)
X_train_scaled.to_csv('../data/processed/X_train_scaled.csv', index=False)
X_test_scaled.to_csv('../data/processed/X_test_scaled.csv', index=False)

# Targets
y_train.to_csv('../data/processed/y_train_multiclass.csv', index=False)
y_test.to_csv('../data/processed/y_test_multiclass.csv', index=False)
y_train_bin.to_csv('../data/processed/y_train_binary.csv', index=False)
y_test_bin.to_csv('../data/processed/y_test_binary.csv', index=False)

print("Processed feature sets and targets saved successfully to data/processed/")

---
## Summary for Co-Analysts

| Component | Decision / Output |
|---|---|
| **Engineered Features** | `pulse_pressure`, `is_hypertensive`, `bmi_category`, `high_waist_risk`, `sleep_category`, `sedentary_high_bmi`, `lifestyle_risk_score` |
| **Leakage Control** | Excluded `fasting_blood_sugar` and `hba1c_level` from pre-screening feature matrix |
| **Categorical Encoding** | One-Hot for nominal features; Binary for `family_history_diabetes` |
| **Target Encodings** | Multiclass (0, 1, 2) + Binary Triage (High Risk = 1, Other = 0) |
| **Split Strategy** | 80/20 Stratified Train/Test split |
| **Scaling** | `StandardScaler` fitted on `X_train` only; exported unscaled (for trees) & scaled (for linear/distance models) |
| **Outputs Saved** | Stored in `data/processed/` for direct consumption in `06_modeling.ipynb` |

**Next Step ->** Open `notebooks/06_modeling.ipynb` to train baseline models, optimize hyperparameters, and evaluate high-risk patient recall.